# Data Loading and Preprocessing

This notebook loads the raw Kaggle data, inspects it, cleans up the types, and saves everything as Parquet for the next notebooks.

## Workflow

1. Import libraries
2. Load `train.csv` and `test.csv` from `data/old/`
3. Inspect the dataset: column types, basic stats, sample rows
4. Check for duplicates and missing values
5. Encode categorical features as integers and downcast numeric columns
6. Save processed DataFrames to Parquet

## Step 1: Import required libraries

In [1]:
import pandas as pd
import warnings

## Step 2: Load the raw CSV data

In [2]:
df_train = pd.read_csv('data/old/train.csv')
df_test = pd.read_csv('data/old/test.csv')

## Step 3: Inspect the dataset structure

In [3]:
# First 10 rows of the training set
df_train.head(10)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,0,Male,21,1,35.0,0,1-2 Year,Yes,65101.0,124.0,187,0
1,1,Male,43,1,28.0,0,> 2 Years,Yes,58911.0,26.0,288,1
2,2,Female,25,1,14.0,1,< 1 Year,No,38043.0,152.0,254,0
3,3,Female,35,1,1.0,0,1-2 Year,Yes,2630.0,156.0,76,0
4,4,Female,36,1,15.0,1,1-2 Year,No,31951.0,152.0,294,0
5,5,Female,31,1,47.0,1,< 1 Year,No,28150.0,152.0,197,0
6,6,Male,23,1,45.0,1,< 1 Year,No,27128.0,152.0,190,0
7,7,Female,47,1,8.0,0,1-2 Year,Yes,40659.0,26.0,262,1
8,8,Female,26,1,28.0,1,< 1 Year,No,31639.0,152.0,36,0
9,9,Female,66,1,11.0,0,1-2 Year,Yes,2630.0,26.0,125,0


In [4]:
# Column names, data types, non-null counts and memory usage
df_train.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11504798 entries, 0 to 11504797
Data columns (total 12 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   Gender                object 
 2   Age                   int64  
 3   Driving_License       int64  
 4   Region_Code           float64
 5   Previously_Insured    int64  
 6   Vehicle_Age           object 
 7   Vehicle_Damage        object 
 8   Annual_Premium        float64
 9   Policy_Sales_Channel  float64
 10  Vintage               int64  
 11  Response              int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 2.5 GB


In [5]:
# Descriptive statistics for all numerical columns
df_train.describe()

,id,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response
count,1.150480e+07,1.150480e+07,1.150480e+07,1.150480e+07,1.150480e+07,1.150480e+07,1.150480e+07,1.150480e+07,1.150480e+07
mean,5.752398e+06,3.838356e+01,9.980220e-01,2.641869e+01,4.629966e-01,3.046137e+04,1.124254e+02,1.638977e+02,1.229973e-01
std,3.321149e+06,1.499346e+01,4.443120e-02,1.299159e+01,4.986289e-01,1.645475e+04,5.403571e+01,7.997953e+01,3.284341e-01
min,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,2.630000e+03,1.000000e+00,1.000000e+01,0.000000e+00
25%,2.876199e+06,2.400000e+01,1.000000e+00,1.500000e+01,0.000000e+00,2.527700e+04,2.900000e+01,9.900000e+01,0.000000e+00
50%,5.752398e+06,3.600000e+01,1.000000e+00,2.800000e+01,0.000000e+00,3.182400e+04,1.510000e+02,1.660000e+02,0.000000e+00
75%,8.628598e+06,4.900000e+01,1.000000e+00,3.500000e+01,1.000000e+00,3.945100e+04,1.520000e+02,2.320000e+02,0.000000e+00
max,1.150480e+07,8.500000e+01,1.000000e+00,5.200000e+01,1.000000e+00,5.401650e+05,1.630000e+02,2.990000e+02,1.000000e+00


In [6]:
# Dataset dimensions: (number of rows, number of columns)
df_train.shape

(11504798, 12)

## Step 4: Assess data quality

In [7]:
# Count duplicate rows (0 means no duplicates)
df_train.duplicated().sum()

0

In [8]:
# Count missing values per column (0 for all columns means the dataset is complete)
df_train.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

## Step 5: Encode categorical features and downcast types

Some columns are stored as strings or 64-bit floats which wastes memory. Here we map string categories (`Gender`, `Vehicle_Age`, `Vehicle_Damage`) to integers and downcast all numeric columns to the smallest type that fits.

In [ ]:
def refactor_data(df):
    df = df.copy()

    try:
        # Encode binary / ordinal string columns as integer codes
        df['Gender'] = df['Gender'].replace({'Male': 0, 'Female': 1}).astype('int8')
        df['Vehicle_Age'] = df['Vehicle_Age'].replace({'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}).astype('int8')
        df['Vehicle_Damage'] = df['Vehicle_Damage'].replace({'No': 0, 'Yes': 1}).astype('int8')

        # Downcast numeric columns to compact integer types
        df['Age'] = df['Age'].astype('int8')
        df['Driving_License'] = df['Driving_License'].astype('int8')
        df['Region_Code'] = df['Region_Code'].astype('int8')
        df['Previously_Insured'] = df['Previously_Insured'].astype('int8')
        df['Annual_Premium'] = df['Annual_Premium'].astype('int32')
        df['Policy_Sales_Channel'] = df['Policy_Sales_Channel'].astype('int16')
        df['Vintage'] = df['Vintage'].astype('int16')
        df['Response'] = df['Response'].astype('int8')

        print(df.info(memory_usage='deep'))

    except KeyError as e:
        # The test set does not contain the target column
        print(f"Column not found: {e}")
    except Exception as e:
        print(f"An error occurred during type conversion: {e}")

    return df

In [15]:
warnings.filterwarnings("ignore")

# Apply the refactoring pipeline to both the training and test sets
df_train_final = refactor_data(df_train)
df_test_final = refactor_data(df_test)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11504798 entries, 0 to 11504797
Data columns (total 12 columns):
 #   Column                Dtype
---  ------                -----
 0   id                    int64
 1   Gender                int8 
 2   Age                   int8 
 3   Driving_License       int8 
 4   Region_Code           int8 
 5   Previously_Insured    int8 
 6   Vehicle_Age           int8 
 7   Vehicle_Damage        int8 
 8   Annual_Premium        int32
 9   Policy_Sales_Channel  int16
 10  Vintage               int16
 11  Response              int8 
dtypes: int16(2), int32(1), int64(1), int8(8)
memory usage: 263.3 MB
None
Column not found: 'Response'


In [11]:
# Preview the processed training set
df_train_final

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,0,0,21,1,35,0,1,1,65101,124,187,0
1,1,0,43,1,28,0,2,1,58911,26,288,1
2,2,1,25,1,14,1,0,0,38043,152,254,0
3,3,1,35,1,1,0,1,1,2630,156,76,0
4,4,1,36,1,15,1,1,0,31951,152,294,0
...,...,...,...,...,...,...,...,...,...,...,...,...
11504793,11504793,0,48,1,6,0,1,1,27412,26,218,0
11504794,11504794,1,26,1,36,0,0,1,29509,152,115,1
11504795,11504795,1,29,1,32,1,0,0,2630,152,189,0
11504796,11504796,1,51,1,28,0,1,1,48443,26,274,1


In [12]:
# Preview the processed test set
df_test_final

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage
0,11504798,1,20,1,47,0,0,0,2630,160,228
1,11504799,0,47,1,28,0,1,1,37483,124,123
2,11504800,0,47,1,43,0,1,1,2630,26,271
3,11504801,1,22,1,47,1,0,0,24502,152,115
4,11504802,0,51,1,19,0,1,0,34115,124,148
...,...,...,...,...,...,...,...,...,...,...,...
7669861,19174659,0,57,1,28,0,1,1,51661,124,109
7669862,19174660,0,28,1,50,1,0,0,25651,152,184
7669863,19174661,0,47,1,33,1,1,0,2630,138,63
7669864,19174662,0,30,1,28,0,0,1,38866,124,119


## Step 6: Save to Parquet

Save the cleaned DataFrames as Parquet files so `eda.ipynb` and `model.ipynb` can load them quickly.

In [13]:
def save_to_parquet(df, filepath):
    """Save a DataFrame to a Parquet file and print a confirmation message."""
    df.to_parquet(filepath, index=False)
    print(f"Saved {len(df):,} rows to '{filepath}'")

# Step 6: Export both processed datasets
save_to_parquet(df_train_final, 'data/processed/train_final.parquet')
save_to_parquet(df_test_final, 'data/processed/test_final.parquet')

Saved 11,504,798 rows to 'data/processed/train_final.parquet'
Saved 7,669,866 rows to 'data/processed/test_final.parquet'
